<a href="https://colab.research.google.com/github/Aaguilar123/Macro-Liquidity-Asset-Leaderboard-Forecast/blob/Aaguilar123-patch-1/shared_eda/Data_pull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import yfinance as yf
import pandas as pd
import pandas_datareader.data as web
from datetime import datetime

# 1. Project Parameters
START_DATE = '2015-01-01'
END_DATE = datetime.today().strftime('%Y-%m-%d')

# The 5 Market Assets
YAHOO_TICKERS = ['QQQ', 'XLP', 'GLD', 'BTC-USD', 'UUP']

# The 3 Core Macro Indicators:
FRED_TICKERS = ['FEDFUNDS', 'CPIAUCSL', 'UNRATE']

def pull_market_data():
    print(f"Pulling Market Data from Yahoo Finance to calculate Dollar Volume...")
    data = yf.download(YAHOO_TICKERS, start=START_DATE, end=END_DATE)
    prices = data['Close'].ffill()
    volume = data['Volume'].ffill()


    dollar_volume = volume.copy()
    # 1. ETFs are in shares -> Multiply by Price
    etfs = ['QQQ', 'XLP', 'GLD', 'UUP']
    for etf in etfs:
        dollar_volume[etf] = prices[etf] * volume[etf]

    # 2. Bitcoin is ALREADY in Dollars
    dollar_volume['BTC-USD'] = volume['BTC-USD']

    return dollar_volume

def pull_macro_data():
    print(f"Pulling Macro Data from FRED ({START_DATE} to {END_DATE})...")
    # FRED API
    macro_data = web.DataReader(FRED_TICKERS, 'fred', START_DATE, END_DATE)

    # Clean up column names for the team
    macro_data.rename(columns={
        'FEDFUNDS': 'Fed_Rate',
        'CPIAUCSL': 'CPI_Inflation',
        'UNRATE': 'Unemployment_Rate'
    }, inplace=True)

    # Force the index name to match Yahoo's exactly
    macro_data.index.name = 'Date'

    return macro_data

# 2. Execute Data Pull & Export

if __name__ == "__main__":
    df_volume = pull_market_data()
    df_macro = pull_macro_data()
    # Export to CSV.
    print("Exporting data to CSV files...")

    df_volume.to_csv('market_volume.csv')
    df_macro.to_csv('macro_indicators.csv')

    print("The 2 CSV files are ready.")

Pulling Market Data from Yahoo Finance to calculate Dollar Volume...


/tmp/ipykernel_3851/3452233756.py:18: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(YAHOO_TICKERS, start=START_DATE, end=END_DATE)
[*********************100%***********************]  5 of 5 completed


Pulling Macro Data from FRED (2015-01-01 to 2026-04-15)...
Exporting data to CSV files...
The 2 CSV files are ready.
